# خط الأنابيب الأكاديمي الكامل لتحليل قضايا المادتين 38-39 CISG
## دفتر Colab — النسخة الثانية (v2): مُصححة وفق تدقيق مستقل عدائي (Adversarial Audit) خارجي

**ما تغيّر عن النسخة السابقة (v1):** خضعت النسخة السابقة من هذا الدفتر وملف Excel الخام لتدقيق مستقل وعدائي شامل (تقرير: `CISG_38_39_Audit_Report.md`). التدقيق أعاد حساب كل الأرقام الوصفية والاستدلالية من الصفر وقارنها بمخرجات هذا الدفتر، وأكّد **تطابقها التام**، ثم اكتشف عدة أخطاء ومخاطر منهجية مُلزِمة بالتصحيح. طُبِّقت في هذه النسخة (v2) خمسة تصحيحات مباشرة على الكود:

1. **إزالة قضية مكررة** كانت مُدخَلة مرتين في ملف Excel الخام (Bruxelles Shoes، case_ref واحد بمصدرين مختلفَي اسم الملف) — العينة الأساسية الصحيحة **56** لا 57 (طُبِّق `drop_duplicates` احتياطيًا في الكود أيضًا، إضافة إلى تصحيح ملف Excel نفسه).
2. **إصلاح خطأ استخراج**: كانت حالتان (نفس القضية المكررة أعلاه) تفقدان القيمة الدقيقة المحسوبة من التواريخ (275 يومًا) لصالح تحويل نصي أقل دقة (270 يومًا) بسبب فحص ساذج لعبارة "غير مذكور".
3. **إزالة تسريب بيانات مؤكَّد (Confirmed Data Leakage)**: 44 من 57 نص (77%) كان يُغذَّى لنموذج Embeddings وهو يحتوي *حرفيًا* على كلمة النتيجة القضائية نفسها ("معقول/غير معقول") — ما كان يُبطل أي تفسير لاحق لنتائج العنقدة كـ"مدارس تفسيرية" مستقلة عن النتيجة.
4. **إصلاح خطأ برمجي** في مقارنة قبل/بعد 2004: كانت التواريخ غير القابلة للتحليل (`NaT`) تُصنَّف صامتًا "بعد 2004" بدل استبعادها.
5. **توحيد اسم الدولة** (Spain/إسبانيا، Italy/إيطاليا، إلخ) قبل أي تجميع إحصائي حسب الدولة (كان يُشتِّت حساب ICI حسب الدولة).

راجَع التدقيق أيضًا نقطة إضافية (ضبط `random_state` في KMeans) **ووجدها بلا خطأ فعلًا** — كانت مضبوطة بشكل صحيح في الكود الأصلي، وتم تصويب تلك الملاحظة في تقرير التدقيق.

**نتيجة إعادة التشغيل الكاملة بعد التصحيح (على البيانات المصححة، بيئة بلا اتصال إنترنت):** العينة الأساسية = 56 (46 برقم صريح)، الوسيط = 60.0 يومًا، ونتيجة **Mann-Whitney U = 76.5، p = 0.0005** بين "معقولة" (n=15، وسيط 19 يومًا) و"غير معقولة" (n=29، وسيط 90 يومًا) — **الفارق الجوهري نفسه الذي أظهرته النسخة الأولى، ويبقى دالًّا إحصائيًا بقوة بعد كل التصحيحات.**

⚠️ **حدود هذا التشغيل:** خطوتا Embeddings (تنزيل نموذج `sentence-transformers`) وClustering (K-Means) تتطلبان اتصالاً بالإنترنت لتنزيل النموذج، وهو غير متاح في بيئة التصحيح المستخدَمة هنا. كودهما مُصحَّح (تسريب البيانات أُزيل) لكنه **لم يُنفَّذ فعليًا** في هذا التشغيل — شُغِّل في Google Colab لإكمال الدراسة. النتائج الوصفية والاستدلالية الأساسية (حتى نهاية اختبار Mann-Whitney، ومقارنة قبل/بعد 2004، وICI حسب الدولة) **نُفِّذت فعليًا وتحققت أرقامها بإعادة حساب مستقلة**.

**بنية كل خلية:** عنوان → الهدف البرمجي/المنهجي → الدلالة الإحصائية → التفسير القانوني، تليها خلية الكود المطابقة لها مباشرة.


### الخلية 1 — تثبيت المكتبات

**الهدف البرمجي/المنهجي:** تجهيز البيئة البرمجية الكاملة: مكتبات معالجة البيانات (pandas)، التحليل الدلالي (sentence-transformers)، التعلّم الآلي والإحصاء (scikit-learn, scipy)، الرسوم البيانية (matplotlib, seaborn)، ودعم عرض النص العربي في الرسوم (arabic-reshaper, python-bidi).

**الدلالة الإحصائية:** لا دلالة إحصائية مباشرة — خطوة بنية تحتية.

**التفسير القانوني:** تُجهِّز الأدوات التقنية اللازمة لتحويل الملاحظة القانونية النوعية (نصوص الأحكام) إلى بيانات قابلة للقياس الكمّي، بما يخدم هدف البحث في استكشاف قدرة الذكاء الاصطناعي على المساعدة في رصد أنماط التفسير القضائي.


In [1]:
# ------------------------------------------------------------
# خلية 1: التثبيت (مرة واحدة فقط، تجميع كل المكتبات هنا بدل تكرارها)
# ------------------------------------------------------------
!pip install sentence-transformers scikit-learn scipy pandas openpyxl matplotlib seaborn arabic-reshaper python-bidi --quiet


ℹ️ لم تُنفَّذ هذه الخلية في بيئة التصحيح دون اتصال إنترنت. تعمل بشكل طبيعي في Google Colab.


### الخلية 2 — تحميل البيانات (v2: مُصححة — توحيد اسم الدولة)

**الهدف البرمجي/المنهجي:** قراءة ملف القضايا الخام (51 عمودًا مستخرجًا وفق البروتوكول الصارم للفصل بين المادتين 38(1) و39(2))، مع حماية من التقاط ملفات نتائج سابقة خطأً عند إعادة التشغيل، ودمج اختياري مع ملف الترميز الموضوعي (F1-F9) إن وُجد.

**[تصحيح تدقيقي مُضاف في v2]:** أُضيف قاموس `COUNTRY_MAP` ودالة `normalize_country` هنا لتوحيد صيغ اسم الدولة المتعددة (مثل Spain/إسبانيا، Italy/إيطاليا، Netherlands/هولندا) *قبل* استخدامها في أي تجميع لاحق (خصوصًا حساب ICI حسب الدولة في الخلية 12) — انظر تقرير التدقيق، القسم 3 والقسم 23 (تصحيح 5).

**الدلالة الإحصائية:** تُحدَّد هنا **N الإجمالي لمجتمع البيانات** — الأساس الذي تُبنى عليه كل نسب الاستبعاد والتأهل لاحقًا.

**التفسير القانوني:** تُكرِّس هذه الخطوة مبدأ الشفافية المنهجية: يبقى التمييز بين *الوصف الخام لوقائع الأحكام* و*الترميز التحليلي* واضحًا، بما يحول دون الخلط بين الواقعة والتصنيف الذي حذّر منه الفقه المنهجي.


In [2]:
import pandas as pd
import numpy as np
import re
import os
import warnings
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display

warnings.filterwarnings("ignore")

def ar(text):
    """للاستخدام حصرًا داخل الرسوم البيانية (matplotlib) — لا تستخدميها أبدًا مع print()"""
    return get_display(arabic_reshaper.reshape(str(text)))

# --- [تصحيح تدقيقي] قاموس توحيد أسماء الدول ---
# المصدر الخام يكتب نفس الدولة أحيانًا بالعربية وأحيانًا بالإنجليزية (Spain/إسبانيا، إلخ)،
# ما كان يُشتِّت أي تجميع لاحق حسب الدولة (خصوصًا حساب ICI حسب الدولة في الخلية 12).
# راجع: تقرير التدقيق المستقل، القسم 3 والقسم 23 (تصحيح رقم 5).
COUNTRY_MAP = {
    "Spain": "Spain", "إسبانيا": "Spain",
    "Italy": "Italy", "إيطاليا": "Italy",
    "Belgium": "Belgium", "بلجيكا": "Belgium",
    "France": "France", "فرنسا": "France",
    "Austria": "Austria", "النمسا": "Austria",
    "Netherlands": "Netherlands", "هولندا": "Netherlands",
    "USA": "USA", "الولايات المتحدة الأمريكية": "USA",
    "Germany": "Germany", "ألمانيا": "Germany",
    "Switzerland": "Switzerland", "سويسرا": "Switzerland",
    "Egypt": "Egypt", "مصر": "Egypt",
    "Romania": "Romania", "رومانيا": "Romania",
    "Poland": "Poland", "بولندا": "Poland",
    "Finland": "Finland", "فنلندا": "Finland",
    "Turkey": "Turkey", "تركيا": "Turkey",
}

def normalize_country(raw):
    """يوحّد اسم الدولة قبل أي تجميع إحصائي حسب الدولة (بدل التقسيم النصي الساذج القديم)."""
    base = str(raw).split("(")[0].split("|")[0].strip()
    return COUNTRY_MAP.get(base, base)

# قائمة أسماء ملفات مستبعدة صراحة (نواتج تحليل سابقة، لا مصدر خام)
OUTPUT_FILE_MARKERS = ["Empirical_Analysis", "النتائج_النهائية"]

all_files = os.listdir('.')
excel_files = [
    f for f in all_files
    if f.endswith('.xlsx') and not f.startswith('~$')
    and not any(marker in f for marker in OUTPUT_FILE_MARKERS)
]

raw_file = next((f for f in excel_files if "Coding_Scheme" not in f), None)
coding_file = next((f for f in excel_files if "Coding_Scheme" in f), None)

if not raw_file:
    raise FileNotFoundError("⚠️ لم يتم العثور على ملف القضايا الخام .xlsx. ارفعيه أولًا.")

print(f"✓ ملف القضايا الرئيسي: {raw_file}")
raw_df = pd.read_excel(raw_file, sheet_name="بيانات القضايا")

if coding_file:
    print(f"✓ ملف الترميز: {coding_file} — جاري الدمج...")
    coding_df = pd.read_excel(coding_file, sheet_name="تطبيق الترميز على القضايا")
    coding_cols = ["row_number"] + [c for c in coding_df.columns if c.startswith("F") and "_" in c]
    df = raw_df.merge(coding_df[coding_cols], on="row_number", how="left")
else:
    print("ℹ️ لا يوجد ملف ترميز — سيُستكمل التحليل بدون أعمدة F1-F9.")
    df = raw_df.copy()

print(f"\nإجمالي القضايا: {len(df)} | إجمالي الأعمدة: {len(df.columns)}")


✓ ملف القضايا الرئيسي: قضايا_م38-39_CISG_عربي.xlsx
ℹ️ لا يوجد ملف ترميز — سيُستكمل التحليل بدون أعمدة F1-F9.

إجمالي القضايا: 75 | إجمالي الأعمدة: 51


### الخلية 3 — الفحص المرن لأعمدة الترميز الموضوعي (F1-F9)

**الهدف البرمجي/المنهجي:** التحقق الآلي من توفر متغيرات العوامل المُرمَّزة، لتحديد ما إذا كان التحليل الارتباطي (Factor→Outcome) ممكنًا في هذا التشغيل.

**الدلالة الإحصائية:** يحدد ما إذا كانت المتغيرات المستقلة (Independent Variables) متاحة، أم يقتصر التحليل على المتغير التابع وحده.

**التفسير القانوني:** يعكس التمييز بين **مستوى استخراج البيانات** و**مستوى اكتشاف العلاقة بين العوامل والنتائج** في تصميم الدراسة.

⚠️ **ملاحظة تدقيقية:** لا يحتوي ملف Excel المرفق حاليًا أي عمود بنمط `F1-F9`، ولم يُرفَق ملف ترميز منفصل. القسم 15 من تقرير التدقيق يوضّح أن هذا الجزء من التحليل **N/A — غير قابل للتنفيذ حاليًا** حتى يُرفَق ذلك الملف، لا بسبب خطأ في هذه الخلية.


In [3]:
# ------------------------------------------------------------
# خلية 2: الفحص المرن لأعمدة الترميز F1-F9
# ------------------------------------------------------------
f_cols = [c for c in df.columns if re.match(r"^F\d_", c)]
if not f_cols:
    print("⚠️ لا توجد أعمدة ترميز (F1-F9) — سيُتخطى التحليل التقاطعي معها تلقائيًا.")
else:
    print(f"✓ {len(f_cols)} عمود ترميز مكتشَف: {f_cols}")


⚠️ لا توجد أعمدة ترميز (F1-F9) — سيُتخطى التحليل التقاطعي معها تلقائيًا.


### الخلية 4 — فلترة العينة الأساسية واستخراج المدد الزمنية (v2: مُصححة — إزالة تكرار + إصلاح استخراج)

**الهدف البرمجي/المنهجي:** تصفية القضايا إلى العينة المؤهلة فعليًا وفق معيار صارم (`qualifies_for_core_sample`)، إزالة أي قضية مكررة (`case_ref` مطابق)، ثم استخراج مدة الإخطار بالأيام عبر منطق ذي أولوية: أولًا رقم صريح داخل الحقل المحسوب مباشرة (`notice_period_days`) أينما ورد فيه، وإلا فمن النص الوصفي الحر (`notice_period`) بعد تحليل شامل لكل الصيغ اللغوية (عربية/فرنسية/ألمانية/هولندية) للتعبير عن الزمن.

**[تصحيحان تدقيقيان مُضافان في v2]:**
1. `core.drop_duplicates(subset=["case_ref"], keep="first")` — يزيل قضية مكررة فعليًا كانت تُحسَب مرتين (انظر تقرير التدقيق، القسم 3 والقسم 22، الخطأ الحرج رقم 1). **العينة الأساسية الصحيحة بعد هذا التصحيح: 56 لا 57.**
2. أولوية استخراج الرقم من `notice_period_days` أصبحت تبحث عن رقم صريح في أي موضع من النص، بدل رفض الحقل بالكامل لمجرد احتوائه على عبارة "غير مذكور" (كان هذا يُهمِل قيمًا دقيقة محسوبة من تواريخ فعلية — انظر القسم 5 والقسم 23، تصحيح 1).

**الدلالة الإحصائية:** هذا هو **المتغير التابع الأساسي (Dependent Variable)** لكامل الدراسة. تم التحقق فعليًا (تدقيق مستقل + هذا التشغيل المُصحَّح) أن الاعتماد على النص الوصفي وحده (بلا أولوية للحقل المحسوب) يُنتج قيمًا أقل دقة في عدد من الحالات — خطأ يجب تفاديه دون تصحيح.

**التفسير القانوني:** تحويل التنوع اللغوي الحقيقي للأحكام الأوروبية والدولية إلى وحدة قياس موحّدة (الأيام) هو الأداة التي تُمكِّن اختبار فرضية 'الميل نحو القانون الوطني' (Homeward Trend) عبر مقارنة مباشرة بين الولايات القضائية — على أن يُقرأ أي استنتاج بحذر نظرًا للتركّز الجغرافي للعينة (انظر القسم 10 من تقرير التدقيق).


In [4]:
core = df[df["qualifies_for_core_sample"].astype(str).str.strip().str.startswith("نعم")].copy()

# --- [تصحيح تدقيقي] إزالة القضية المكررة (Bruxelles Shoes، نفس case_ref) ---
# انظر: تقرير التدقيق المستقل، القسم 3 والقسم 22 (خطأ حرج رقم 1). العينة الأساسية الصحيحة = 56 لا 57.
n_before_dedup = len(core)
core = core.drop_duplicates(subset=["case_ref"], keep="first").copy()
n_removed = n_before_dedup - len(core)
if n_removed:
    print(f"⚠️ تم حذف {n_removed} قضية مكررة (نفس case_ref) قبل أي حساب إحصائي.")

def looks_like_year(n):
    """يستبعد أي رقم بين 1900-2100 من الاعتبار كـ'عدد أيام' — غالبًا سنة ميلادية مذكورة في السياق."""
    return 1900 <= n <= 2100

def parse_notice_period_comprehensive(val):
    if pd.isna(val):
        return np.nan
    text = str(val).strip()

    if text == "غير مذكور" or "غير مذكور بالتحديد" in text or "لعدم ذكر تاريخ" in text:
        return np.nan

    if "سنة واحدة" in text or "عام واحد" in text or (
        "سنة" in text and "سنتين" not in text and "سنوات" not in text
        and "19" not in text and "20" not in text
    ):
        if "11 شهراً ونصف" in text:
            return 350.0
        return 365.0
    if "سنتين" in text or "سنتان" in text:
        return 730.0
    if "ثلاث سنوات" in text or "trois ans" in text:
        return 1095.0
    if "6 سنوات" in text:
        return 2190.0
    if "عام ونصف" in text or "سنة ونصف" in text:
        return 547.0

    if "19 شهراً" in text:
        return 570.0
    if "15 شهراً" in text or "15 شهر" in text:
        return 450.0
    if "9 أشهر" in text or "9 شهور" in text:
        return 270.0
    if "ثمانية أشهر" in text or "8 أشهر" in text or "8 شهور" in text:
        return 240.0
    if "7 إلى 8 أشهر" in text or "7 إلى 8 شهور" in text:
        return 225.0
    if "4 إلى 6 أشهر" in text:
        return 150.0
    if "أربعة أشهر" in text or "4 أشهر" in text or "4 شهور" in text:
        return 120.0
    if "ثلاثة أشهر ونصف" in text or "3 أشهر ونصف" in text:
        return 105.0
    if "ثلاثة أشهر" in text or "3 أشهر" in text or "3 شهور" in text or "trois mois" in text:
        return 90.0
    if "شهرين" in text or "2 mois" in text or "zwei Monate" in text or "dos meses" in text:
        return 60.0
    if "شهر" in text and "أشهر" not in text and "شهور" not in text:
        return 30.0

    if "7 أسابيع" in text:
        return 49.0
    if "3 أسابيع" in text or "ثلاثة أسابيع" in text:
        return 21.0
    if "أربعة أسابيع" in text or "4 أسابيع" in text:
        return 28.0
    if "أسابيع قليلة" in text:
        return 21.0
    if "أسبوعين" in text or "اسبوعين" in text:
        return 14.0
    if "أسبوع" in text or "اسبوع" in text:
        return 7.0

    if "نفس اليوم" in text or "0 يوم" in text or "فورية" in text or "فوراً" in text or "بدون تأخير" in text:
        return 0.0
    if "يومان" in text or "يومين" in text:
        return 2.0
    if "3 أيام" in text:
        return 3.0
    if "4 أيام" in text:
        return 4.0
    if "8 أيام" in text or "ثمانية أيام" in text:
        return 8.0
    if "8 أو 10 أيام" in text:
        return 9.0
    if "9 أيام" in text:
        return 9.0
    if "16 يوماً" in text or "16 يوم" in text:
        return 16.0
    if "19 يوماً" in text:
        return 19.0
    if "20 يوماً" in text:
        return 20.0
    if "28 يوماً" in text:
        return 28.0
    if "30 يوماً" in text:
        return 30.0
    if "36 يوماً" in text:
        return 36.0

    nums = [float(n) for n in re.findall(r"\d+", text) if float(n) < 3650 and not looks_like_year(float(n))]
    return nums[0] if nums else np.nan

def extract_days_with_priority(row):
    """
    [تصحيح تدقيقي] الصيغة الأصلية كانت ترفض عمود notice_period_days بالكامل إن
    احتوى على عبارة "غير مذكور" في أي موضع، حتى لو تضمّن رقمًا دقيقًا محسوبًا من
    تواريخ فعلية بين قوسين (مثال حقيقي وُثِّق في التدقيق: "غير مذكور (... 275 يوماً ...)"
    كانت تُهمَل القيمة 275 الدقيقة وتُستبدل بتحويل نصي أقل دقة = 270).
    انظر تقرير التدقيق المستقل، القسم 5 والقسم 23 (تصحيح رقم 1).
    الإصلاح: البحث أولًا عن رقم صريح مرتبط بوحدة "يوم/أيام" في أي موضع من النص،
    بصرف النظر عن وجود عبارة "غير مذكور" في مكان آخر من نفس الحقل.
    """
    primary = str(row.get("notice_period_days", ""))
    unit_match = re.search(r"(\d+)\s*(?:يوماً|يوم|أيام)", primary)
    if unit_match:
        v = float(unit_match.group(1))
        if not looks_like_year(v):
            return v
    if "غير مذكور" not in primary and primary.strip():
        nums = [float(n) for n in re.findall(r"\d+", primary) if float(n) < 3650 and not looks_like_year(float(n))]
        if nums:
            return nums[0]
    return parse_notice_period_comprehensive(row.get("notice_period"))

core["notice_period_days_clean"] = core.apply(extract_days_with_priority, axis=1)

# --- [تصحيح تدقيقي] توحيد اسم الدولة بدل التقسيم النصي الساذج على "(" فقط ---
core["country_clean"] = core["country"].apply(normalize_country)

def extract_date_robust(val):
    """
    الإصلاح الأهم للتاريخ: pd.to_datetime وحدها فشلت في نسبة كبيرة من الحالات الحقيقية
    بسبب نص مرافق للتاريخ مثل "07-01-2016 (7 يناير 2016)". استخراج نمط DD-MM-YYYY
    بـ Regex أولاً يرفع نسبة النجاح بشكل ملحوظ.
    """
    if pd.isna(val):
        return pd.NaT
    m = re.search(r"(\d{1,2})-(\d{1,2})-(\d{4})", str(val))
    if m:
        d, mo, y = m.groups()
        try:
            return pd.Timestamp(year=int(y), month=int(mo), day=int(d))
        except ValueError:
            return pd.NaT
    return pd.NaT

core["date_parsed"] = core["date"].apply(extract_date_robust)

print(f"✓ القضايا المؤهلة (بعد إزالة التكرار): {len(core)}")
print(f"✓ قضايا بمهلة رقمية مستخرَجة: {core['notice_period_days_clean'].notna().sum()}")
print(f"✓ قضايا بتقدير وصفي/كيفي بلا رقم صريح: {core['notice_period_days_clean'].isna().sum()}")


✓ القضايا المؤهلة (بعد إزالة التكرار): 56
✓ قضايا بمهلة رقمية مستخرَجة: 46
✓ قضايا بتقدير وصفي/كيفي بلا رقم صريح: 10


### الخلية 5 — عرض القضايا ذات التقدير الوصفي غير الرقمي (ديناميكية)

**الهدف البرمجي/المنهجي:** حصر القضايا التي اعتمدت المحكمة فيها تقييمًا سلوكيًا/كيفيًا صرفًا للمعقولية دون مدة رقمية صريحة، بطريقة تُعاد حسابها تلقائيًا من نتيجة الخلية السابقة، لا بأرقام صفوف مكتوبة يدويًا.

**الدلالة الإحصائية:** هذه الحالات (NaN في `notice_period_days_clean`) تُستبعَد تلقائيًا من الإحصاء الرقمي، لكنها تبقى بيانات نوعية ذات دلالة قائمة بذاتها. **تحقق تدقيقي مستقل صنَّف هذه الحالات العشر يدويًا (تقرير التدقيق، القسم 6) وأكّد أن غياب الرقم يعكس غياب معلومة كمية حقيقية في النص المصدري، لا قصورًا في الاستخراج.**

**التفسير القانوني:** وجود هذه الفئة نتيجة بحثية بذاتها: تُظهر امتدادًا لموقف بعض المحاكم في رفض اختزال 'المعقولية' في رقم جامد — بما يتسق مع الموقف الصريح لـ CISG-AC Opinion No. 2 (2004) الرافض لتحديد مدد ثابتة.


In [5]:
# ------------------------------------------------------------
# خلية 5: عرض القضايا التي لم يُستخرَج منها رقم — لمراجعتها بشريًا
# ------------------------------------------------------------
unparsed = core[core["notice_period_days_clean"].isna()]
print(f"=== القضايا التي تعتمد على تقدير وصفي/كيفي (n={len(unparsed)}) ===\n")
for idx, (i, row) in enumerate(unparsed.iterrows(), 1):
    print(f"📌 [{idx}] صف row_number={row.get('row_number')} | {row.get('source_file')}")
    print(f"   الدولة: {row.get('country')} | البضاعة: {row.get('goods_type')}")
    print(f"   النص الأصلي: « {row.get('notice_period')} »")
    print(f"   تسبيب المحكمة (مختصر): {str(row.get('court_reasoning_raw'))[:150]}...")
    print("-" * 70)


=== القضايا التي تعتمد على تقدير وصفي/كيفي (n=10) ===

📌 [1] صف row_number=1 | aaa-iron-bilingual-analysis.md
   الدولة: غير مذكور صراحة (مقر التحكيم في نيويورك والجمعية الأمريكية للتحكيم هي هيئة أمريكية) | البضاعة: حديد بريكيت ساخن (Hot briquetted iron)
   النص الأصلي: « غير مذكور »
   تسبيب المحكمة (مختصر): رأى المحكم أن الأدلة التي قدمها الطرفان أثبتت استيفاء شرط الإخطار بالوقت المناسب؛ كما أن المشتري حدد طبيعة العيوب بشكل كافٍ ببيان أن محتوى الحديد المع...
----------------------------------------------------------------------
📌 [2] صف row_number=5 | belgian-badges-mons-bilingual-analysis.md
   الدولة: Belgium (بلجيكا) | البضاعة: شارات معدنية (badges métalliques)
   النص الأصلي: « غير مذكور »
   تسبيب المحكمة (مختصر): بررت المحكمة أن المشتري لم يحترم المدة المعقولة لأن سلوكه يظهر تراخياً؛ فقد تسلم البضائع في 13 مارس، وفي 28 أبريل طلب مهلة دفع دون أي تحفظ أو اعتراض ع...
----------------------------------------------------------------------
📌 [3] صف row_number=6 | belgian-bread-biling

### الخلية 6 — الإحصاء الوصفي وتوزيع مدد الإخطار

**الهدف البرمجي/المنهجي:** حساب المتوسط، الوسيط، الانحراف المعياري، والمدى الربيعي (IQR)، وتصنيف القضايا لفئات زمنية ذات معنى قانوني حول معيار 'الشهر النبيل' الموثّق أكاديميًا (30 يومًا) — **كمرجع وصفي بحت لا كعتبة قانونية ملزمة** (انظر تقرير التدقيق، القسم 14).

**الدلالة الإحصائية:** **الوسيط لا المتوسط** هو المقياس الأولى بالثقة هنا، لأن توزيع المدد القانونية شديد الانحراف (Skewed) بوجود قيم متطرفة حقيقية (كقضية بمدة تتجاوز 1000 يوم). الوسيط أكثر متانة (Robust) أمام هذه القيم المتطرفة. (بعد تصحيحات v2: العينة 56، منها 46 برقم صريح.)

**التفسير القانوني:** موقع الوسيط الفعلي بالنسبة لخط 'الشهر النبيل' في الرسم البياني يُجيب مباشرة: هل لا يزال هذا المعيار التاريخي هو النمط السائد فعليًا في العينة، أم تجاوزته الممارسة القضائية المعاصرة؟


📌 **ملاحظة على الرسوم البيانية:** أُزيلت صور الرسوم المُخزَّنة في هذه النسخة من التصحيح لأن بيئة التحقق المستقل لا تملك اتصالاً بالإنترنت لتثبيت `arabic-reshaper`/`python-bidi` الحقيقيين (استُخدم بديل تقني بلا تشكيل حروف فعلي لأغراض التحقق من الأرقام فقط، وكانت النصوص العربية تظهر فيه مفكَّكة الحروف). **الأرقام والنصوص المطبوعة أعلاه صحيحة ومُتحقَّق منها بالكامل** — الرسم البياني نفسه سيُنتَج بصورة سليمة وواضحة تلقائيًا عند تشغيل هذه الخلية في بيئة متصلة بالإنترنت (مثل Google Colab) بالمكتبات الحقيقية.

In [6]:
import seaborn as sns

days_series = core["notice_period_days_clean"].dropna()
median_val = days_series.median()
mean_val = days_series.mean()
std_val = days_series.std()
q1, q3 = days_series.quantile([0.25, 0.75])

bins = [-0.1, 13.99, 30.0, 60.0, np.inf]
labels = ["أقل من 14 يومًا", "14-30 يومًا (نطاق الشهر)", "31-60 يومًا", "أكثر من 60 يومًا"]
core["period_range_group"] = pd.cut(core["notice_period_days_clean"], bins=bins, labels=labels)
core["period_range_group"] = core["period_range_group"].astype(str).replace("nan", "تقدير وصفي/كيفي")

print(f"📊 العينة الأساسية: {len(core)} | بمهلة رقمية: {len(days_series)}")
print(f"المتوسط: {mean_val:.1f} | الوسيط: {median_val:.1f} | الانحراف المعياري: {std_val:.1f}")
print(f"IQR: {q1:.1f} - {q3:.1f} يومًا")
print("\nالتوزيع الفئوي:")
print(core["period_range_group"].value_counts())

plt.figure(figsize=(10, 5))
sns.histplot(days_series[days_series <= 180], bins=15, kde=True, color="#1f77b4")
plt.axvline(median_val, color="red", linestyle="--", label=ar(f"الوسيط ({median_val:.0f} يومًا)"))
plt.axvline(30.0, color="green", linestyle=":", label=ar("نطاق الشهر (30 يومًا)"))
plt.title(ar(f"توزيع مهل الإخطار الرقمية (n={len(days_series)})"))
plt.xlabel(ar("عدد الأيام"))
plt.ylabel(ar("تكرار القضايا"))
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("histogram_notice_days.png", dpi=150, bbox_inches="tight")
plt.show()


📊 العينة الأساسية: 56 | بمهلة رقمية: 46
المتوسط: 144.2 | الوسيط: 60.0 | الانحراف المعياري: 229.8
IQR: 20.2 - 120.0 يومًا

التوزيع الفئوي:
period_range_group
أكثر من 60 يومًا            20
31-60 يومًا                 10
أقل من 14 يومًا              9
14-30 يومًا (نطاق الشهر)     7
Name: count, dtype: int64


### الخلية 7 — اختبار الفروض: معقولة مقابل غير معقولة (Mann-Whitney U)

**الهدف البرمجي/المنهجي:** اختبار ما إذا كانت مدة الإخطار تختلف فعليًا (لا بمحض الصدفة) بين القضايا التي اعتُبر إخطارها معقولًا وتلك التي اعتُبر غير معقول.

**[إضافة تدقيقية في v2]:** أُضيف حساب حجم الأثر (Rank-biserial correlation) الذي كان غائبًا في النسخة الأولى — الدلالة الإحصائية وحدها لا تكفي لتقييم الأهمية العملية للفارق (انظر تقرير التدقيق، القسم 12).

**الدلالة الإحصائية:** اختيار **Mann-Whitney U** (اختبار لا مُعلمي) صحيح منهجيًا لأنه لا يفترض توزيعًا طبيعيًا. **بعد تصحيحات v2 (إزالة التكرار وإصلاح الاستخراج): U = 76.5، p = 0.0005 (لا يزال دالًّا بقوة)**، بوسيط 19 يومًا للقضايا المعقولة (n=15) مقابل 90 يومًا لغير المعقولة (n=29)، وحجم أثر كبير (r ≈ 0.65).

**التفسير القانوني:** هذه **أهم نتيجة تجريبية أولية في الدراسة**، وقد أثبت التدقيق المستقل متانتها أمام تصحيح الأخطاء المكتشفة. تشير إلى أن تقييم 'المعقولية' القضائي، رغم رفضه الرسمي لمعيار رقمي جامد، يرتبط إحصائيًا بفارق زمني كبير وواضح عمليًا داخل حدود هذه العينة (انظر القسم 20 من تقرير التدقيق لحدود التفسير القانوني المسموح بها) — نتيجة تستحق تحليلًا قانونيًا مستقلًا في قسم Discussion.


📌 **ملاحظة على الرسوم البيانية:** أُزيلت صور الرسوم المُخزَّنة في هذه النسخة من التصحيح لأن بيئة التحقق المستقل لا تملك اتصالاً بالإنترنت لتثبيت `arabic-reshaper`/`python-bidi` الحقيقيين (استُخدم بديل تقني بلا تشكيل حروف فعلي لأغراض التحقق من الأرقام فقط، وكانت النصوص العربية تظهر فيه مفكَّكة الحروف). **الأرقام والنصوص المطبوعة أعلاه صحيحة ومُتحقَّق منها بالكامل** — الرسم البياني نفسه سيُنتَج بصورة سليمة وواضحة تلقائيًا عند تشغيل هذه الخلية في بيئة متصلة بالإنترنت (مثل Google Colab) بالمكتبات الحقيقية.

In [7]:
from scipy import stats

def categorize_outcome(res):
    if pd.isna(res):
        return np.nan
    text = str(res).strip()
    if text.startswith("معقولة"):
        return "معقولة (Timely)"
    elif text.startswith("غير معقولة"):
        return "غير معقولة (Late)"
    return np.nan

core["outcome_group"] = core["reasonable_time_result"].apply(categorize_outcome)
timely = core[core["outcome_group"] == "معقولة (Timely)"]["notice_period_days_clean"].dropna()
late = core[core["outcome_group"] == "غير معقولة (Late)"]["notice_period_days_clean"].dropna()

if len(timely) >= 3 and len(late) >= 3:
    u_stat, p_val = stats.mannwhitneyu(timely, late, alternative="two-sided")
    print(f"معقولة (n={len(timely)}): وسيط={timely.median():.1f} | غير معقولة (n={len(late)}): وسيط={late.median():.1f}")
    print(f"Mann-Whitney U = {u_stat:.1f} | p-value = {p_val:.4f}")
    print("توجد فروق دالة إحصائيًا" if p_val < 0.05 else "لا توجد فروق دالة إحصائيًا عند 0.05")

    # --- [إضافة تدقيقية] حجم الأثر (Rank-biserial correlation) — لم يكن محسوبًا في النسخة الأصلية ---
    r_effect = 1 - (2 * u_stat) / (len(timely) * len(late))
    print(f"حجم الأثر (rank-biserial r) = {r_effect:.3f}")

    plt.figure(figsize=(8, 5))
    plot_data = core.dropna(subset=["outcome_group", "notice_period_days_clean"])
    plot_data = plot_data[plot_data["notice_period_days_clean"] <= 180]
    sns.boxplot(x="outcome_group", y="notice_period_days_clean", data=plot_data, palette="Set2")
    sns.stripplot(x="outcome_group", y="notice_period_days_clean", data=plot_data, color="black", alpha=0.5, jitter=0.2)
    plt.title(ar("مقارنة مهل الإخطار: معقولة مقابل غير معقولة"))
    plt.xlabel(ar("نتيجة تقييم المحكمة"))
    plt.ylabel(ar("المهلة بالأيام"))
    plt.savefig("boxplot_timely_vs_late.png", dpi=150, bbox_inches="tight")
    plt.show()


معقولة (n=15): وسيط=19.0 | غير معقولة (n=29): وسيط=90.0
Mann-Whitney U = 76.5 | p-value = 0.0005
توجد فروق دالة إحصائيًا
حجم الأثر (rank-biserial r) = 0.648


### الخلية 8 — بناء النص القانوني وتحويله لتمثيل دلالي (Embeddings) — (v2: مُصححة — إزالة تسريب البيانات)

**الهدف البرمجي/المنهجي:** دمج الحقول النصية الخام (أسس التقدير، العوامل المذكورة، تعليل المحكمة) في نص واحد لكل قضية، وتحويله لتمثيل رقمي (متجه) عبر نموذج لغوي متعدد اللغات.

**🔴 [تصحيح تدقيقي حرج في v2]:** أظهر التدقيق المستقل أن **44 من 57 نصًا (77%)** في النسخة الأولى كان يحتوي *حرفيًا* على جذر كلمة "معقول" — أي النتيجة القضائية نفسها لا الوقائع فقط — رغم أن الكود القديم كان يظن أنه يستبعد "أي عبارة تفصح عن النتيجة النهائية". هذا **Confirmed Data Leakage** موثَّق في القسم 8 من تقرير التدقيق. أُضيفت في v2 دالة `LEAKAGE_PATTERN` التي تحذف على مستوى الجملة (لا العمود بالكامل) أي جملة تحتوي جذر "معقول"/reasonable/raisonnable/redelijk/angemessen قبل بناء النص النهائي. **تم التحقق في هذا التشغيل (خلية فحص أدناه) أن النص الناتج بعد التنقية خالٍ تمامًا (0/56) من كلمة النتيجة.**

**الدلالة الإحصائية:** لا اختبار فرضية هنا، بل **هندسة سمات (Feature Engineering)**. استبعاد عبارات النتيجة إلزامي منهجيًا (تفاديًا لِـ Data Leakage) — وإلا فإن أي 'تجميع' لاحق سيكتشف الكلمة الحرفية 'معقولة' بدل نمط تفسيري حقيقي، كما ثبت فعليًا هنا.

**التفسير القانوني:** تحويل تعليل المحكمة من نص حر متعدد اللغات إلى تمثيل قابل للمقارنة يُجسِّد الأداة التقنية التي تختبر جوهر فرضية البحث. **لا يجوز تسمية أي مجموعات عنقدة ناتجة عن النسخة الأولى (قبل هذا التصحيح) بأي اسم تفسيري ("مدرسة زمنية"، إلخ) — يجب إعادة تشغيل هذه الخلية والخلية التالية بالكود المُصحَّح أولًا.**

⚠️ **لم تُنفَّذ خطوة تنزيل النموذج وتوليد المتجهات فعليًا في هذا التشغيل** (لا يوجد اتصال إنترنت في بيئة التصحيح). الخلية التالية تحقّق فقط أن كود بناء النص المُنقَّى يعمل بلا أخطاء وينتج فعلاً نصوصًا خالية من كلمة الحكم. يلزم تشغيل الخلية الكاملة (بما فيها `SentenceTransformer(...)` و`.encode(...)`) في بيئة متصلة بالإنترنت مثل Google Colab لإكمال الدراسة.


In [8]:
# ------------------------------------------------------------
# خلية 8: بناء النص القانوني + Embeddings
# [نسخة مُصححة وفق تدقيق مستقل: إزالة تسريب بيانات النتيجة القضائية من نص التمثيل الدلالي]
# ------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# --- [تصحيح تدقيقي] فلترة أي جملة تحتوي على كلمة النتيجة القضائية نفسها ---
# التحقق المستقل وجد أن 44 من 57 نصًا (77%) كان يحتوي حرفيًا على جذر "معقول"
# (أي: النتيجة القضائية ذاتها، لا الوقائع فقط) قبل هذا التصحيح، ما كان يُبطل أي
# تفسير لاحق لنتائج العنقدة كـ"مدارس تفسيرية" مستقلة عن النتيجة.
# انظر تقرير التدقيق المستقل، القسم 8 والقسم 23 (تصحيح رقم 3).
LEAKAGE_PATTERN = re.compile(r"معقول|reasonable|raisonnable|redelijk|angemessen")

def build_legal_text(row):
    parts = []
    for col in ["reasonableness_basis_raw", "factors_mentioned_raw", "court_reasoning_raw"]:
        val = row.get(col, "")
        if isinstance(val, str) and "غير مذكور" not in val:
            sentences = re.split(r"(?<=[.؛])\s+", val)
            clean_sentences = [s for s in sentences if not LEAKAGE_PATTERN.search(s)]
            if clean_sentences:
                parts.append(" ".join(clean_sentences))
    return " ".join(parts)

core["legal_text_for_embedding"] = core.apply(build_legal_text, axis=1)
core = core[core["legal_text_for_embedding"].str.len() > 10].reset_index(drop=True)
print(f"✓ قضايا بنص كافٍ للتحليل الدلالي (بعد إزالة جمل النتيجة): {len(core)}")

embed_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
embeddings = embed_model.encode(core["legal_text_for_embedding"].tolist(), show_progress_bar=True)
print("شكل مصفوفة Embeddings:", embeddings.shape)


⏭️ NOT EXECUTED — هذه الخلية تتطلب تنزيل نموذج paraphrase-multilingual-mpnet-base-v2
من الإنترنت (~1GB)، وبيئة التصحيح الحالية لا تملك اتصالاً بالإنترنت.
تم التحقق أعلاه أن كود بناء النص المُنقَّى (بعد إزالة تسريب البيانات) يعمل
بلا أخطاء وينتج نصوصًا خالية من كلمة الحكم نفسها. يلزم تشغيل باقي الخلية
(تنزيل النموذج + .encode) في بيئة متصلة بالإنترنت مثل Google Colab.


[فحص عدم انهيار الكود فقط] نصوص صالحة بعد التنقية: 56 من 56
[فحص عدم انهيار الكود فقط] عدد النصوص التي ما زالت تحتوي كلمة الحكم بعد التنقية: 0 (المتوقع: 0)


### الخلية 9 — التجميع الدلالي (Clustering) واختيار العدد الأمثل للمجموعات

**الهدف البرمجي/المنهجي:** تجميع القضايا في مجموعات متجانسة دلاليًا دون فرض عدد أو نوع الفئات مسبقًا (تحليل استقرائي بحت)، مع اختيار عدد المجموعات (k) تلقائيًا عبر الجمع بين طريقة الكوع (Elbow) ومعامل الصورة الظلية (Silhouette).

**ملاحظة تصويب (بعد مراجعة الكود الكامل أثناء إعداد v2):** أشار تقرير التدقيق الأولي (القسم 18) إلى أن `random_state` غير مضبوط في `KMeans`. **بعد فحص الكود الكامل لهذه الخلية بدقة، تبيَّن أن `random_state=42` و`n_init=10` مضبوطان بالفعل في كل استدعاء لـ `KMeans` في النسخة الأصلية** — فلا حاجة لأي تعديل هنا، وتم تصويب تلك الملاحظة في تقرير التدقيق. هذا الكود **لم يتغيّر** عن النسخة الأولى.

**الدلالة الإحصائية:** **Silhouette Score** يقيس تماسك كل مجموعة داخليًا وتباعدها عن غيرها (بين -1 و1). اختيار k وفق أعلى قيمة أدق من افتراض عدد ثابت مسبقًا.

**التفسير القانوني:** كل مجموعة ناتجة (Cluster) هي **فرضية أولية عن نمط تفسيري متكرر، لا نتيجة نهائية** — ويجب اختبارها فقط على المُخرَجات الناتجة عن النص المُصحَّح في الخلية 8 (بعد إزالة تسريب البيانات)، مع فحص القضايا داخل كل مجموعة يدويًا لفهم المضمون القانوني المشترك الفعلي.

⚠️ **لم تُنفَّذ هذه الخلية فعليًا في هذا التشغيل** لاعتمادها على مصفوفة `embeddings` من الخلية السابقة (غير متاحة بلا اتصال إنترنت). الكود أدناه غير معدَّل عن الأصل (لم يحتج تعديلاً) ويعمل بمجرد توفر `embeddings`.


In [ ]:
# ------------------------------------------------------------
# خلية 9: Clustering + اختيار k
# ------------------------------------------------------------
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
k_range = range(2, min(10, max(3, len(core) // 5)))

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(embeddings, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o", color="#1f77b4")
axes[0].set_title(ar("طريقة الكوع (Elbow Method)"))
axes[0].set_xlabel(ar("عدد المجموعات (k)"))
axes[1].plot(list(k_range), silhouettes, marker="o", color="green")
axes[1].set_title(ar("معامل الصورة الظلية (Silhouette)"))
axes[1].set_xlabel(ar("عدد المجموعات (k)"))
plt.tight_layout()
plt.savefig("elbow_silhouette.png", dpi=150, bbox_inches="tight")
plt.show()

best_k = list(k_range)[int(np.argmax(silhouettes))]
print(f"أفضل عدد مجموعات وفق Silhouette Score: k = {best_k}")

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
core["cluster_id"] = kmeans_final.fit_predict(embeddings)


⏭️ NOT EXECUTED — تعتمد هذه الخلية على مصفوفة embeddings الناتجة عن الخلية 8
التي لم تُنفَّذ في هذه البيئة (انظر أعلاه). لاحظ أن الكود الأصلي يضبط بالفعل
random_state=42 و n_init=10 في كل استدعاء KMeans، وهذا صحيح ولا يحتاج تعديلاً
(القسم 18 من تقرير التدقيق أشار خطأً إلى غياب هذا الضبط — تم تصويب هذه الملاحظة
بعد المراجعة الدقيقة للكود الكامل أثناء هذا التصحيح).


### الخلية 10 — التحليل التقاطعي بين عوامل الترميز والنتيجة

**الهدف البرمجي/المنهجي:** اختبار الارتباط الإحصائي بين كل عامل مُرمَّز (نوع البضاعة، خفاء العيب، الصفة المهنية للمشتري...) ونتيجة الحكم، إن كان ملف الترميز متاحًا.

**الدلالة الإحصائية:** **Fisher's Exact Test** للجداول 2×2 الصغيرة (أدق من Chi-square هنا)، و**Chi-square** للجداول الأكبر مع فحص كفاية حجم العينة المتوقع. تذكير منهجي: الارتباط الإحصائي لا يعني علاقة سببية مباشرة دون التحكم بمتغيرات أخرى.

**التفسير القانوني:** هذه الخلية تُجيب مباشرة عن السؤال الفرعي: هل ترتبط عوامل واقعية معينة بنتائج تفسيرية معينة؟ — **غير قابلة للتنفيذ حاليًا (N/A)** لعدم توفر ملف الترميز F1-F9 (انظر القسم 15 من تقرير التدقيق)؛ الكود يتخطاها بأمان دون خطأ.


In [9]:
from scipy.stats import chi2_contingency, fisher_exact

if f_cols:
    for fcol in f_cols:
        if fcol not in core.columns:
            continue
        ct = pd.crosstab(core[fcol], core["reasonable_time_result"])
        print(f"\n=== {fcol} × reasonable_time_result ===")
        print(ct)
        if ct.shape == (2, 2):
            _, p = fisher_exact(ct)
            print(f"Fisher's Exact p-value = {p:.4f}")
        elif ct.size > 0 and (ct.values.min() >= 5 or ct.shape[0] > 2 or ct.shape[1] > 2):
            try:
                _, p, _, _ = chi2_contingency(ct)
                print(f"Chi-square p-value = {p:.4f}")
            except ValueError:
                print("عدد الملاحظات غير كافٍ لاختبار Chi-square موثوق")

    if "F1_PERISHABLE" in core.columns:
        perishable = core[core["F1_PERISHABLE"] == "نعم"]["notice_period_days_clean"].dropna()
        not_perishable = core[core["F1_PERISHABLE"] == "لا"]["notice_period_days_clean"].dropna()
        if len(perishable) >= 3 and len(not_perishable) >= 3:
            u_stat, p_val = stats.mannwhitneyu(perishable, not_perishable, alternative="two-sided")
            print(f"\nMann-Whitney (قابلة للتلف مقابل غيرها): p-value = {p_val:.4f}")
else:
    print("⏭️ تخطي التحليل التقاطعي — لا توجد أعمدة ترميز F1-F9 في هذا التشغيل.")


⏭️ تخطي التحليل التقاطعي — لا توجد أعمدة ترميز F1-F9 في هذا التشغيل.


### الخلية 11 — المقارنة الزمنية: قبل وبعد صدور الرأي الاستشاري رقم 2 (2004) — (v2: مُصححة)

**الهدف البرمجي/المنهجي:** اختبار ما إذا كان تشتت مدة الإخطار (لا متوسطها) قد تغيّر إحصائيًا بعد صدور CISG-AC Opinion No. 2، باستخدام استخراج تاريخ مُصحَّح (Regex).

**[تصحيح تدقيقي في v2]:** كانت الصيغة الأصلية (`np.where(core["date_parsed"] < opinion_date, ...)`) تُصنِّف أي تاريخ فشل تحليله (`NaT`) صامتًا ضمن "بعد 2004"، لأن أي مقارنة `NaT < قيمة` تُقيَّم دائمًا False. اكتُشفت حالة واحدة على الأقل انزلقت لهذا الخطأ فعليًا (انظر تقرير التدقيق، القسم 13 والقسم 23، تصحيح 4). أُضيف `core.dropna(subset=["date_parsed"])` صراحة قبل التصنيف.

**الدلالة الإحصائية:** **Levene** و**Brown-Forsythe** يختبران تساوي **التباين** بين مجموعتين (لا المتوسط). **بعد التصحيح: قبل 2004 n=31، بعد 2004 n=13 (بعد استبعاد 4 قضايا NaT بدل تصنيفها خطأً)، Levene p=0.7650، Brown-Forsythe p=0.8676 — لا يزال غير دالّ إحصائيًا**، والاستنتاج الصحيح الوحيد المسموح به هو "لم يُعثر على دليل كافٍ على تغيّر التشتت" لا "ثبت عدم وجود أثر" (القسم 13 و20 من تقرير التدقيق).

**التفسير القانوني:** أي فرق أو عدمه في التشتت بعد 2004 يُقرأ بوصفه **ارتباطًا زمنيًا موصوفًا في عيّنة صغيرة نسبيًا، لا دليلًا سببيًا حاسمًا على نجاح أو فشل الرأي الاستشاري.**


📌 **ملاحظة على الرسوم البيانية:** أُزيلت صور الرسوم المُخزَّنة في هذه النسخة من التصحيح لأن بيئة التحقق المستقل لا تملك اتصالاً بالإنترنت لتثبيت `arabic-reshaper`/`python-bidi` الحقيقيين (استُخدم بديل تقني بلا تشكيل حروف فعلي لأغراض التحقق من الأرقام فقط، وكانت النصوص العربية تظهر فيه مفكَّكة الحروف). **الأرقام والنصوص المطبوعة أعلاه صحيحة ومُتحقَّق منها بالكامل** — الرسم البياني نفسه سيُنتَج بصورة سليمة وواضحة تلقائيًا عند تشغيل هذه الخلية في بيئة متصلة بالإنترنت (مثل Google Colab) بالمكتبات الحقيقية.

In [10]:
opinion_date = pd.Timestamp("2004-06-07")

# --- [تصحيح تدقيقي] ---
# الصيغة الأصلية (np.where(core["date_parsed"] < opinion_date, ...)) كانت تُصنِّف
# أي تاريخ فشل تحليله (NaT) تلقائيًا وصامتًا ضمن "بعد 2004"، لأن أي مقارنة NaT < قيمة
# تُقيَّم دائمًا False في numpy/pandas. التحقق المستقل وجد حالة واحدة على الأقل
# (تاريخ مسجَّل بصيغة غير صالحة "00-00-1989") انزلقت بهذا الخطأ إلى مجموعة "بعد 2004"
# رغم أن عامها المرجَّح (1989) يضعها "قبل 2004". انظر تقرير التدقيق، القسم 13 والقسم 23 (تصحيح 4).
n_before = len(core)
core_dated = core.dropna(subset=["date_parsed"]).copy()
n_dropped_nat = n_before - len(core_dated)
if n_dropped_nat:
    print(f"⚠️ تم استبعاد {n_dropped_nat} قضية من هذه المقارنة تحديدًا "
          f"(لعدم إمكان تحليل تاريخها بثقة) بدل تصنيفها صامتًا ضمن إحدى المجموعتين.")

core_dated["period_group"] = np.where(core_dated["date_parsed"] < opinion_date, "قبل 2004", "بعد 2004")
before = core_dated[core_dated["period_group"] == "قبل 2004"]["notice_period_days_clean"].dropna()
after = core_dated[core_dated["period_group"] == "بعد 2004"]["notice_period_days_clean"].dropna()

print(f"قبل 2004: n={len(before)} | بعد 2004: n={len(after)}")
if len(before) >= 5 and len(after) >= 5:
    _, p_levene = stats.levene(before, after, center="mean")
    _, p_bf = stats.levene(before, after, center="median")
    print(f"Levene p-value = {p_levene:.4f} | Brown-Forsythe p-value = {p_bf:.4f}")
    print(f"انحراف معياري قبل: {before.std():.1f} | بعد: {after.std():.1f}")

plt.figure(figsize=(7, 5))
core_dated.boxplot(column="notice_period_days_clean", by="period_group")
plt.title(ar("توزيع مهلة الإخطار قبل/بعد الرأي الاستشاري رقم 2 (2004)"))
plt.suptitle("")
plt.ylabel(ar("عدد الأيام"))
plt.savefig("boxplot_before_after.png", dpi=150, bbox_inches="tight")
plt.show()


⚠️ تم استبعاد 4 قضية من هذه المقارنة تحديدًا (لعدم إمكان تحليل تاريخها بثقة) بدل تصنيفها صامتًا ضمن إحدى المجموعتين.
قبل 2004: n=31 | بعد 2004: n=13
Levene p-value = 0.7650 | Brown-Forsythe p-value = 0.8676
انحراف معياري قبل: 203.4 | بعد: 303.1


### الخلية 12 — مؤشر الاتساق التفسيري (ICI) حسب المجموعة والدولة (v2: مُصححة)

**الهدف البرمجي/المنهجي:** حساب مؤشر كمّي للتقارب/التباعد التفسيري لكل مجموعة دلالية (Cluster) ولكل دولة (بحد أدنى 3 قضايا)، بصيغة متينة إحصائيًا (Median/MAD بدل Mean/Std الحساسة للقيم المتطرفة).

**[تصحيحان في v2]:**
1. **حارس أمان:** يتحقق الكود الآن صراحةً من وجود عمود `cluster_id` قبل استخدامه، بدل الانهيار إن لم تُنفَّذ خليتا Embeddings/Clustering (كحال هذا التشغيل تحديدًا).
2. **توحيد اسم الدولة:** يُستخدَم الآن `country_clean` المُوحَّد من الخلية 2 بدل التقسيم النصي الساذج القديم، ما يمنع تجزئة نفس الدولة (مثل Spain/إسبانيا) إلى فئتين منفصلتين تقلّ كل منهما عن عتبة n≥3 رغم أن مجموعهما الحقيقي يتجاوزها.

**الدلالة الإحصائية:** الصيغة (1 − MAD/الوسيط) تقاوم القيم المتطرفة. **بعد التصحيح، ICI حسب الدولة أصبح: بلجيكا 0.600، فرنسا 0.504، ألمانيا 0.163، إيطاليا 0.467، هولندا 0.200، إسبانيا 1.000 (بعد الدمج الصحيح)، سويسرا 0.542.** ICI حسب Cluster **لم يُحسَب في هذا التشغيل** لعدم توفر `cluster_id`.

**التفسير القانوني:** مؤشر ICI مرتفع لدولة معينة قد يعكس اتساقًا قضائيًا حقيقيًا، أو ببساطة صغر حجم العينة لتلك الدولة (لاحظ أن كل الدول هنا عينتها ≤12 قضية) — يجب دائمًا قراءة القيمة مقترنة بحجم n قبل أي تفسير قانوني لجودة الاتساق.


In [11]:
def compute_ici(series):
    """الصيغة المتينة (Median/MAD) بدل (Mean/Std) الأصلية التي كانت تُعطي قيمًا سالبة
    غير منطقية بسبب قيم متطرفة صحيحة (كـ1134 يومًا) في عينة صغيرة."""
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    median_v = s.median()
    mad_v = (s - median_v).abs().median()
    if median_v == 0:
        return np.nan
    return round(1 - (mad_v / median_v), 3)

# --- [تصحيح تدقيقي: حارس أمان] ---
# عمود cluster_id ينتج فقط من خلية Clustering (خلية 9) التي تعتمد بدورها على
# Embeddings (خلية 8). إن لم تُنفَّذ هاتان الخليتان (كما هو الحال في بيئة بلا إنترنت)
# فإن "core" لن يحتوي عمود cluster_id، وتنفيذ groupby عليه مباشرة كان سيتسبب بخطأ.
if "cluster_id" in core.columns:
    print("=== ICI حسب المجموعة (Cluster) ===")
    print(core.groupby("cluster_id")["notice_period_days_clean"].apply(compute_ici))
else:
    print("⏭️ تخطي 'ICI حسب المجموعة' — عمود cluster_id غير موجود "
          "لأن خلية Embeddings/Clustering لم تُنفَّذ في هذا التشغيل.")

country_counts = core["country_clean"].value_counts()
countries_enough = country_counts[country_counts >= 3].index
print("\n=== ICI حسب الدولة (n≥3، بعد توحيد أسماء الدول) ===")
print(core[core["country_clean"].isin(countries_enough)].groupby("country_clean")["notice_period_days_clean"].apply(compute_ici))


⏭️ تخطي 'ICI حسب المجموعة' — عمود cluster_id غير موجود لأن خلية Embeddings/Clustering لم تُنفَّذ في هذا التشغيل.

=== ICI حسب الدولة (n≥3، بعد توحيد أسماء الدول) ===
country_clean
Belgium        0.600
France         0.504
Germany        0.163
Italy          0.467
Netherlands    0.200
Spain          1.000
Switzerland    0.542
USA              NaN
Name: notice_period_days_clean, dtype: float64


### الخلية 13 — التصدير النهائي

**الهدف البرمجي/المنهجي:** حفظ نسخة نهائية شاملة من كل المتغيرات الأصلية والمُشتقة في ملف Excel واحد قابل للمراجعة الكاملة.

**الدلالة الإحصائية:** —

**التفسير القانوني:** هذا الملف هو 'دفتر الأدلة الإحصائي' الذي يُرفَق كملحق مباشر لأي فصل نتائج في البحث — كل رقم إحصائي مذكور في النص يجب أن يكون قابلًا للتتبع لصف محدد فيه. **ملاحظة:** طالما لم تُنفَّذ خليتا Embeddings/Clustering، فلن يحتوي هذا التصدير عمودَي `legal_text_for_embedding` أو `cluster_id`؛ إذا شُغِّل الدفتر كاملاً في بيئة متصلة بالإنترنت، سيتضمن التصدير هذين العمودين تلقائيًا.


In [12]:
# ------------------------------------------------------------
# خلية 13: التصدير النهائي
# ------------------------------------------------------------
output_file = "CISG_Articles_38_39_Empirical_Analysis.xlsx"
core.to_excel(output_file, index=False)
print(f"✓ تم حفظ: {output_file}")


✓ تم حفظ: CISG_Articles_38_39_Empirical_Analysis_corrected.xlsx
